Replay mode uses recorded fixtures and does not call a live model. Set `NORTHSTAR_MODE=live` with `GEMINI_API_KEY` to opt in, or use `NORTHSTAR_MODE=record` to save synthetic responses.

In [ ]:
from pathlib import Path
from northstar.runtime import get_client, PromptRequest, Message
from lab04 import run_lab

client = get_client(Path("fixtures/replays.json"))  # NORTHSTAR_MODE=replay|live|record

# 04 — Structured Outputs and Typed Interfaces

## Scenario

Northstar converts support requests into typed case briefs. A schema makes a response parseable, but the application must still validate evidence and bound repair.

## Demonstration 1: Syntax Guarantees

The recorded response conforms to the `CaseBrief` schema.

In [ ]:
results = run_lab(client)
assert results["syntax_valid"].numerator == 1

## Demonstration 2: Semantic Failure (Hallucination)

The response cites `pol_elite_instant_refund`, which is not approved.

In [ ]:
assert results["unknown_evidence"].numerator == 1

## Demonstration 3: Application-Side Validation and Bounded Repair

The first repair is rejected; the second uses `NONE`. Exhausted repair terminates in human review.

In [ ]:
assert results["repair_attempts"] == 2
assert results["repair_terminal"] == "valid"
assert results["exhausted"] is None
assert results["exhausted_terminal"] == "human_review"

## Demonstration 4: schema drift

A truncated replay yields `not_json` without raising an exception.

In [ ]:
assert results["malformed_error"] == "not_json"

## Takeaway

Typed output guarantees shape, not truth or authorization. Keep semantic validation and repair limits in application code.

## References

- [Core concepts and workflow](README.md#core-concepts--workflow)
- [Production best practices](README.md#production-best-practices)